# 08 — RT-DETR-L Tuning (Kaggle)

**Goal:** controlled hyperparameter tuning of the fourth object detector, without overwriting `04_RTDETR_Baseline`. This notebook uses the **same 4,062 frames**, **GroupKFold by clip**, **six epochs**, **batch 2**, and **COCOeval** (original `iscrowd`, maxDets=100) for comparability. New trials **start from official pretrained `rtdetr-l.pt`**, not from a fold-0 checkpoint.

**GPU budget:** RT-DETR-L is expensive. **No long training cells run automatically**. Start with a single trial. Two learning-rate candidates are offered; `train_lr5e5` is a low-LR trial and `train_lr2e4` a higher-LR trial. After fold-0 evaluation, pick only one configuration for folds 1–2. An optional 12-epoch experiment is separated and must not be presented as a matched 6-epoch comparison.

**Leaderboard caveat:** fold validation COCO mAP is *not* Kaggle Public Score. The previous YOLO submission improved to **0.18731**, still below organizer baseline **0.22163**. RT-DETR tuning does not guarantee beating it.

## 1 — Setup and check inputs
Enable a Kaggle GPU and add the competition dataset. Pinning an Ultralytics version close to the original baseline reduces protocol drift. GPU availability and dataset paths are checked before training.

In [1]:
import sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.204","scikit-learn","pycocotools","pyyaml"],check=True)
import os, gc, json, random, time, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, yaml, torch, ultralytics
from sklearn.model_selection import GroupKFold
from ultralytics import RTDETR
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from IPython.display import display

SEED=42; FRAME_STEP=6; EPOCHS=6; BATCH=2
NAMES=['person','bicycle','car','van','truck','tricycle','bus','motor']
CANDIDATES=[Path('/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026'),Path('/kaggle/input/multiple-object-video-detection-hw-2-ucu-ai-2026')]
DATA_ROOT=next((p for p in CANDIDATES if (p/'train/annotations_coco.json').exists()),None)
assert DATA_ROOT is not None,'Add competition dataset under Kaggle Input.'
assert torch.cuda.is_available(),'Enable GPU in notebook settings.'
OUT=Path('/kaggle/working/rtdetr_tuning'); SPLITS=OUT/'splits'; RUNS=OUT/'runs'; EVAL=OUT/'coco_evaluation'
for x in (OUT,SPLITS,RUNS,EVAL):x.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
print('GPU:',torch.cuda.get_device_name(0),'Ultralytics:',ultralytics.__version__,'Data:',DATA_ROOT)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4 Ultralytics: 8.4.204 Data: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026


## 2 — Recreate the **exact baseline video folds**
Reconstructs the original sampling and checks expected 1,357 / 1,356 / 1,349 validation frames. If you have a saved `cv_splits_step6.csv`, comparing it to the generated version is recommended before interpreting fold differences.

In [2]:
with (DATA_ROOT/'train/annotations_coco.json').open() as f: coco_raw=json.load(f)
frames=pd.DataFrame([{'image_id':int(im['id']),'file_name':str(im['file_name']),'clip':Path(im['file_name']).parent.name} for im in coco_raw['images']])
frames=frames.sort_values(['clip','file_name']).reset_index(drop=True)
sampled=frames.loc[frames.groupby('clip',sort=True).cumcount()%FRAME_STEP==0].copy().reset_index(drop=True)
sampled['fold']=-1
for fold,(_,val_idx) in enumerate(GroupKFold(n_splits=3).split(sampled,groups=sampled['clip'])):
    sampled.loc[val_idx,'fold']=fold
assert len(frames)==24201 and len(sampled)==4062,(len(frames),len(sampled))
assert sampled['fold'].value_counts().sort_index().to_dict()=={0:1357,1:1356,2:1349}
for fold in (0,1,2):
    tr=sampled[sampled['fold']!=fold]; va=sampled[sampled['fold']==fold]
    assert set(tr['clip']).isdisjoint(set(va['clip']))
    print(f'fold {fold}: train={len(tr)}, val={len(va)}, train clips={tr["clip"].nunique()}, val clips={va["clip"].nunique()}')
sampled.to_csv(OUT/'cv_splits_step6.csv',index=False)

fold 0: train=2705, val=1357, train clips=37, val clips=19
fold 1: train=2706, val=1356, train clips=37, val clips=19
fold 2: train=2713, val=1349, train clips=38, val clips=18


## 3 — Fast local images and labels
Kaggle `/kaggle/input` is read-only; copying the sampled frames to `/kaggle/working` avoids slow-image-access/cache warnings. This costs local disk space, **not training time**. Copy operations skip already-complete files.

In [3]:
from tqdm.auto import tqdm
LOCAL=Path('/kaggle/working/rtdetr_local_data')
for row in tqdm(sampled.itertuples(index=False),total=len(sampled),desc='Check/copy images and labels'):
    rel=Path(row.file_name)
    assert rel.parts[0]=='images',rel
    src_im=DATA_ROOT/'train'/rel
    src_lb=DATA_ROOT/'train'/'labels'/rel.parent.relative_to('images')/(rel.stem+'.txt')
    dst_im=LOCAL/rel
    dst_lb=LOCAL/'labels'/rel.parent.relative_to('images')/(rel.stem+'.txt')
    for src,dst in ((src_im,dst_im),(src_lb,dst_lb)):
        assert src.is_file(),src
        dst.parent.mkdir(parents=True,exist_ok=True)
        if not dst.is_file() or dst.stat().st_size!=src.stat().st_size:shutil.copy2(src,dst)
fold_yamls={}
for fold in (0,1,2):
    folder=SPLITS/f'fold_{fold}';folder.mkdir(parents=True,exist_ok=True)
    for name,part in [('train',sampled[sampled['fold']!=fold]),('val',sampled[sampled['fold']==fold])]:
        image_paths=[str(LOCAL/Path(x)) for x in part['file_name']]
        (folder/f'{name}.txt').write_text('\n'.join(image_paths)+'\n')
    cfg={'train':str(folder/'train.txt'),'val':str(folder/'val.txt'),'nc':8,'names':NAMES}
    yaml_path=folder/'data.yaml';yaml.safe_dump(cfg,yaml_path.open('w'),sort_keys=False)
    fold_yamls[fold]=yaml_path
print('Fast local dataset and 3 fold YAMLs ready.')

Check/copy images and labels:   0%|          | 0/4062 [00:00<?, ?it/s]

Fast local dataset and 3 fold YAMLs ready.


## 4 — Tuning plan (only one trial at a time)
Baseline settings: RT-DETR-L / AdamW `lr0=1e-4`, `lrf=0.1`, weight decay `1e-4`, 640 px, 6 epochs, batch 2, seed 42. We test **learning rate only**, while keeping the training budget fixed. Only one candidate should be selected for the remaining folds. Experimental scores use common COCOeval below.

In [4]:
TRIALS={
    'lr5e5':{'lr0':5e-5,'epochs':6,'imgsz':640},
    'lr2e4':{'lr0':2e-4,'epochs':6,'imgsz':640},
    # OPTIONAL, different training budget: do NOT label as matched 6-epoch comparison.
    'long12_lr1e4':{'lr0':1e-4,'epochs':12,'imgsz':640},
}
print(pd.DataFrame(TRIALS).T)

                  lr0  epochs  imgsz
lr5e5         0.00005     6.0  640.0
lr2e4         0.00020     6.0  640.0
long12_lr1e4  0.00010    12.0  640.0


## 5 — Safe training and resume function
Each experiment and fold has its own folder. `last.pt` enables resume; completed runs are skipped. The completion marker is written **only after** best-weight validation. No checkpoint from a different fold is loaded. Avoid Kaggle `Save & Run All` if it would re-run all expensive cells.

In [5]:
def run_trial(trial, fold=0):
    assert trial in TRIALS and fold in (0,1,2)
    cfg=TRIALS[trial]
    name=f'{trial}_fold_{fold}'; folder=RUNS/name
    last=folder/'weights'/'last.pt'; best=folder/'weights'/'best.pt'; done=folder/'completed.json'
    if done.is_file() and best.is_file():
        info=json.loads(done.read_text());assert info['epochs']==cfg['epochs']
        print('Completed; skipping:',name);return best
    gc.collect();torch.cuda.empty_cache()
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    start=time.monotonic()
    if last.is_file():
        print('Resuming',name,'from last.pt')
        model=RTDETR(str(last))
        model.train(resume=True,device=0,workers=0)
    else:
        model=RTDETR('rtdetr-l.pt')
        model.train(data=str(fold_yamls[fold]),epochs=cfg['epochs'],imgsz=cfg['imgsz'],batch=BATCH,
                    nbs=2,device=0,workers=0,optimizer='AdamW',lr0=cfg['lr0'],lrf=0.1,
                    weight_decay=0.0001,warmup_epochs=1.0,cos_lr=True,mosaic=0.0,
                    amp=False,deterministic=False,seed=SEED,cache=False,
                    patience=cfg['epochs'],save=True,save_period=1,plots=False,
                    project=str(RUNS),name=name,exist_ok=True)
    minutes=(time.monotonic()-start)/60
    assert best.is_file() and last.is_file(),f'Missing weights for {name}'
    # Verify train history has reached intended epoch count.
    history=folder/'results.csv'
    assert history.is_file(),f'Missing training history: {history}'
    hist=pd.read_csv(history);hist.columns=hist.columns.str.strip()
    assert len(hist)>=cfg['epochs'],f'Incomplete training history: {len(hist)} / {cfg["epochs"]}'
    info={'trial':trial,'fold':fold,'epochs':cfg['epochs'],'imgsz':cfg['imgsz'],
          'lr0':cfg['lr0'],'train_minutes_this_call':round(minutes,2),'checkpoint':str(best)}
    done.write_text(json.dumps(info,indent=2))
    print('TRAINING COMPLETE:',name,'train minutes this call',round(minutes,2))
    del model;gc.collect();torch.cuda.empty_cache()
    return best
print('Training function ready; no training executed.')

Training function ready; no training executed.


## 6 — Run the first candidate (fold 0)
Uncomment **one** training line and run it. The second learning rate is optional if your GPU budget allows. If notebook resumes in a fresh session, setup/data cells must be rerun; persistent checkpoints need attaching from a saved Kaggle Version Output.

In [6]:
best_lr5e5 = run_trial('lr5e5', fold=0)

Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rtdetr_tuning/splits/fold_0/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=5e-05, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=lr5e5_fold_0, nbs=2, nms=None, opset

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/6      3.19G     0.7859      0.664     0.1033         46        640: 100% ━━━━━━━━━━━━ 1353/1353 2.1it/s 10:310.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 340/340 3.0it/s 1:530.3sss
                   all       1357      59035      0.279      0.304      0.209      0.106

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/6      3.19G     0.7445     0.5564     0.0898         88        640: 100% ━━━━━━━━━━━━ 1353/1353 2.2it/s 10:260.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 340/340 3.0it/s 1:530.3sss
                   all       1357      59035       0.33       0.31      0.243      0.122

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        4/6      3.19G     0.7081      0.529    0.08357         27        640: 100% ━━━━━━━━━━━━ 1353/1353 2.2it/s 10:250.4ss
                

In [7]:
best_lr2e4 = run_trial('lr2e4', fold=0)

Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rtdetr_tuning/splits/fold_0/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=lr2e4_fold_0, nbs=2, nms=None, opse

## 7 — Shared COCOeval, with original `iscrowd`
Unlike Ultralytics built-in validation, this reports pycocotools COCO metrics with original COCO annotations, category IDs 1..8, **the exact fold image IDs** and `maxDets=[1,10,100]`. 0-based RT-DETR predictions are mapped to 1-based COCO categories. This is the right metric family for later four-architecture comparison.

In [8]:
GT=COCO(str(DATA_ROOT/'train/annotations_coco.json'))
assert set(int(c['id']) for c in GT.dataset['categories'])==set(range(1,9))

def coco_eval_rtdetr(trial,fold=0,conf=0.001,iou=0.7,max_det=300,predict_batch=2):
    assert trial in TRIALS and fold in (0,1,2)
    best=RUNS/f'{trial}_fold_{fold}'/'weights'/'best.pt'
    assert best.is_file(),f'Train {trial} fold {fold} first.'
    folder=EVAL/f'{trial}_fold_{fold}';folder.mkdir(parents=True,exist_ok=True)
    metric_file=folder/'coco_metrics.json';pred_file=folder/'predictions_coco.json'
    if metric_file.is_file() and pred_file.is_file():
        print('Already evaluated:',metric_file);return json.loads(metric_file.read_text())
    part=sampled[sampled['fold']==fold].sort_values('image_id')
    ids=part['image_id'].astype(int).tolist()
    imgs=[str(LOCAL/Path(x)) for x in part['file_name']]
    model=RTDETR(str(best));detections=[];start=time.monotonic()
    for i in range(0,len(imgs),predict_batch):
        results=model.predict(source=imgs[i:i+predict_batch],imgsz=TRIALS[trial]['imgsz'],
                              device=0,batch=predict_batch,conf=conf,iou=iou,
                              max_det=max_det,verbose=False,save=False,augment=False)
        batch_ids=ids[i:i+predict_batch]
        assert len(results)==len(batch_ids)
        for image_id,r in zip(batch_ids,results):
            if r.boxes is None:continue
            for xyxy,cl,score in zip(r.boxes.xyxy.cpu().numpy(),r.boxes.cls.int().cpu().tolist(),r.boxes.conf.cpu().tolist()):
                x1,y1,x2,y2=map(float,xyxy);w=max(0.0,x2-x1);h=max(0.0,y2-y1)
                if int(cl) not in range(8) or w<=0 or h<=0:continue
                detections.append({'image_id':int(image_id),'category_id':int(cl)+1,
                                   'bbox':[x1,y1,w,h],'score':float(score)})
        if (i//predict_batch+1)%100==0:print('Inference',min(i+predict_batch,len(imgs)),'/',len(imgs))
    elapsed=time.monotonic()-start
    assert detections,'No predictions; check classes and confidence threshold.'
    pred_file.write_text(json.dumps(detections))
    dt=GT.loadRes(detections);ev=COCOeval(GT,dt,iouType='bbox')
    ev.params.imgIds=ids;ev.params.catIds=list(range(1,9));ev.params.maxDets=[1,10,100]
    ev.evaluate();ev.accumulate();ev.summarize();s=ev.stats
    row={'trial':trial,'fold':fold,'imgsz':TRIALS[trial]['imgsz'],
         'mAP50-95':float(s[0]),'mAP50':float(s[1]),'mAP75':float(s[2]),
         'AP_small':float(s[3]),'AP_medium':float(s[4]),'AP_large':float(s[5]),
         'inference_ms_per_image':round(1000*elapsed/len(ids),2),
         'n_images':len(ids),'n_detections':len(detections),
         'evaluator':'COCOeval bbox, original iscrowd, maxDets=100'}
    metric_file.write_text(json.dumps(row,indent=2))
    del model,dt,ev;gc.collect();torch.cuda.empty_cache()
    return row
print('Shared COCOeval function ready.')

loading annotations into memory...
Done (t=5.12s)
creating index...
index created!
Shared COCOeval function ready.


In [9]:
result_1 = coco_eval_rtdetr('lr5e5', fold=0)
result_2 = coco_eval_rtdetr('lr2e4', fold=0)

display(pd.DataFrame([result_1, result_2]))

Inference 200 / 1357
Inference 400 / 1357
Inference 600 / 1357
Inference 800 / 1357
Inference 1000 / 1357
Inference 1200 / 1357
Loading and preparing results...
DONE (t=0.46s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=84.40s).
Accumulating evaluation results...
DONE (t=3.73s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.141
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.279
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.126
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.053
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.117
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.318
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.079
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.235
 Average Recall    

,trial,fold,imgsz,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large,inference_ms_per_image,n_images,n_detections,evaluator
0,lr5e5,0,640,0.141185,0.278572,0.126102,0.052858,0.117257,0.318233,79.33,1357,407100,"COCOeval bbox, original iscrowd, maxDets=100"
1,lr2e4,0,640,0.157884,0.319223,0.137671,0.061818,0.139764,0.314793,75.14,1357,407100,"COCOeval bbox, original iscrowd, maxDets=100"


## 8 — Evaluate and compare fold-0 trials
Run the applicable commented lines **after** their training has completed. The baseline 0.155325 is from Ultralytics built-in validation and is **not** a directly comparable COCOeval number. To compare with baseline precisely, attach its saved `best.pt` and evaluate that checkpoint under the same function/protocol (see next section).

In [10]:
# Examples (uncomment only trials whose training is finished):
# display(pd.DataFrame([coco_eval_rtdetr('lr5e5',fold=0)]))
# display(pd.DataFrame([coco_eval_rtdetr('lr2e4',fold=0)]))

rows=[]
for p in sorted(EVAL.glob('*/coco_metrics.json')):rows.append(json.loads(p.read_text()))
if rows:
    df=pd.DataFrame(rows).sort_values(['fold','mAP50-95'],ascending=[True,False]);display(df)
    df.to_csv(OUT/'trial_comparison.csv',index=False)
else:print('No completed COCO evaluations yet.')

,trial,fold,imgsz,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large,inference_ms_per_image,n_images,n_detections,evaluator
0,lr2e4,0,640,0.157884,0.319223,0.137671,0.061818,0.139764,0.314793,75.14,1357,407100,"COCOeval bbox, original iscrowd, maxDets=100"
1,lr5e5,0,640,0.141185,0.278572,0.126102,0.052858,0.117257,0.318233,79.33,1357,407100,"COCOeval bbox, original iscrowd, maxDets=100"


## 9 — Optional: evaluate the historical 3-fold RT-DETR baseline checkpoints without retraining
Attach `04_RTDETR_Baseline` **Notebook Output** to Kaggle Input. Search for `best.pt` and set paths, then compare the same fold / COCO protocol. The earlier baseline 0.150299 mean was from Ultralytics validator; **do not compare it directly to the new COCOeval**.

In [13]:
from pathlib import Path

BASELINE_BEST = {}

for fold in (0, 1, 2):
    matches = [
        p for p in Path("/kaggle/input").rglob("best.pt")
        if f"rtdetr_fold_{fold}_compact" in str(p)
        and f"fold_{fold}" in p.parts
    ]

    assert len(matches) == 1, (
        f"Expected one checkpoint for fold {fold}, "
        f"found: {matches}"
    )

    BASELINE_BEST[fold] = str(matches[0])

    print(
        f"Fold {fold}: {matches[0]} "
        f"({matches[0].stat().st_size / 1e6:.1f} MB)"
    )

print("All three RT-DETR baseline checkpoints found!")

Fold 0: /kaggle/input/datasets/margopaduchak/rtdetr/rtdetr_fold_0_compact/fold_0/weights/best.pt (66.2 MB)
Fold 1: /kaggle/input/datasets/margopaduchak/rtdetr/rtdetr_fold_1_compact/fold_1/weights/best.pt (66.2 MB)
Fold 2: /kaggle/input/datasets/margopaduchak/rtdetr/rtdetr_fold_2_compact/fold_2/weights/best.pt (66.2 MB)
All three RT-DETR baseline checkpoints found!


In [14]:
def coco_eval_external(checkpoint,fold=0,imgsz=640,tag='historical_baseline'):
    checkpoint=Path(checkpoint);assert checkpoint.is_file()
    folder=EVAL/f'{tag}_fold_{fold}';folder.mkdir(parents=True,exist_ok=True)
    metrics_file=folder/'coco_metrics.json';pred_file=folder/'predictions_coco.json'
    if metrics_file.is_file() and pred_file.is_file():return json.loads(metrics_file.read_text())
    # Use temporary trial metadata; external model is never trained here.
    TRIALS['_baseline_external']={'imgsz':imgsz,'lr0':1e-4,'epochs':6}
    alias_dir=RUNS/f'_baseline_external_fold_{fold}'/'weights';alias_dir.mkdir(parents=True,exist_ok=True)
    # Avoid copying large checkpoints: use a local symlink, supported by Kaggle Linux runtime.
    alias=alias_dir/'best.pt'
    if alias.is_symlink() or alias.is_file():alias.unlink()
    alias.symlink_to(checkpoint)
    result=coco_eval_rtdetr('_baseline_external',fold=fold)
    TRIALS.pop('_baseline_external',None)
    return result

# After filling BASELINE_BEST with existing checkpoint paths, uncomment:
# for fold,path in BASELINE_BEST.items():
#     if path is not None:display(pd.DataFrame([coco_eval_external(path,fold=fold)]))

In [15]:
baseline_results = []

for fold in (0, 1, 2):
    result = coco_eval_external(
        BASELINE_BEST[fold],
        fold=fold,
        imgsz=640,
        tag="historical_baseline"
    )
    baseline_results.append(result)

display(pd.DataFrame(baseline_results))

Inference 200 / 1357
Inference 400 / 1357
Inference 600 / 1357
Inference 800 / 1357
Inference 1000 / 1357
Inference 1200 / 1357
Loading and preparing results...
DONE (t=0.47s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=83.54s).
Accumulating evaluation results...
DONE (t=3.69s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.160
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.319
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.142
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.060
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.137
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.329
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.082
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.243
 Average Recall    

,trial,fold,imgsz,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large,inference_ms_per_image,n_images,n_detections,evaluator
0,_baseline_external,0,640,0.159562,0.319478,0.141852,0.059519,0.137410,0.328711,76.18,1357,407100,"COCOeval bbox, original iscrowd, maxDets=100"
1,_baseline_external,1,640,0.158670,0.307495,0.141774,0.045659,0.145755,0.336940,73.89,1356,406800,"COCOeval bbox, original iscrowd, maxDets=100"
2,_baseline_external,2,640,0.149269,0.289550,0.139299,0.074074,0.147938,0.252157,75.88,1349,404700,"COCOeval bbox, original iscrowd, maxDets=100"


In [16]:
import json
import pandas as pd
from pathlib import Path

rows = []

for name in ("lr5e5", "lr2e4", "_baseline_external"):
    path = EVAL / f"{name}_fold_0" / "coco_metrics.json"

    if path.exists():
        metrics = json.loads(path.read_text())
        rows.append({
            "configuration": name,
            "mAP50-95": metrics["mAP50-95"],
            "AP_small": metrics["AP_small"],
        })
    else:
        print("Missing:", path)

df = pd.DataFrame(rows)

display(df.sort_values("mAP50-95", ascending=False))

,configuration,mAP50-95,AP_small
2,_baseline_external,0.159562,0.059519
1,lr2e4,0.157884,0.061818
0,lr5e5,0.141185,0.052858


## 10 — Selected configuration: train folds 1 and 2 **only after** fold-0 analysis
Do **not** automatically pick the winning model before comparing with the historical baseline. Select the best justified configuration on fold 0, then independently initialize each remaining fold from `rtdetr-l.pt`. This gives 3-fold results for the tuned setup, though fold 0 was used in model selection.

In [17]:
# RT-DETR: LR = 0.0002, fold 1
result_lr2e4_fold1 = run_trial('lr2e4', fold=1)

display(pd.DataFrame([
    coco_eval_rtdetr('lr2e4', fold=1)
]))

New https://pypi.org/project/ultralytics/8.4.205 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rtdetr_tuning/splits/fold_1/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mo

,trial,fold,imgsz,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large,inference_ms_per_image,n_images,n_detections,evaluator
0,lr2e4,1,640,0.154352,0.302678,0.135579,0.046623,0.133897,0.340428,74.24,1356,406800,"COCOeval bbox, original iscrowd, maxDets=100"


In [18]:
# RT-DETR: LR = 0.0002, fold 2
result_lr2e4_fold2 = run_trial('lr2e4', fold=2)

display(pd.DataFrame([
    coco_eval_rtdetr('lr2e4', fold=2)
]))

New https://pypi.org/project/ultralytics/8.4.206 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rtdetr_tuning/splits/fold_2/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mo

,trial,fold,imgsz,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large,inference_ms_per_image,n_images,n_detections,evaluator
0,lr2e4,2,640,0.14533,0.286614,0.13343,0.06515,0.152704,0.227355,75.88,1349,404700,"COCOeval bbox, original iscrowd, maxDets=100"


## 11 — Three-fold comparison (read-only)
Reports selected tuned configuration mean/std from COCOeval. The 6-epoch original baseline must be separately re-evaluated using the same function before numerical baseline–tuned conclusions.

In [19]:
SELECTED_TRIAL = 'lr2e4'
if SELECTED_TRIAL is None:
    print('Select a tuned configuration in section 10 first.')
else:
    paths=[EVAL/f'{SELECTED_TRIAL}_fold_{f}'/'coco_metrics.json' for f in (0,1,2)]
    missing=[str(p) for p in paths if not p.exists()]
    if missing:print('Still missing folds:',missing)
    else:
        df=pd.DataFrame([json.loads(p.read_text()) for p in paths]).sort_values('fold')
        display(df)
        cols=['mAP50-95','mAP50','mAP75','AP_small','AP_medium','AP_large','inference_ms_per_image']
        summary=df[cols].agg(['mean','std']);display(summary)
        df.to_csv(OUT/'tuned_3fold_results.csv',index=False)
        summary.to_csv(OUT/'tuned_3fold_summary.csv')

,trial,fold,imgsz,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large,inference_ms_per_image,n_images,n_detections,evaluator
0,lr2e4,0,640,0.157884,0.319223,0.137671,0.061818,0.139764,0.314793,75.14,1357,407100,"COCOeval bbox, original iscrowd, maxDets=100"
1,lr2e4,1,640,0.154352,0.302678,0.135579,0.046623,0.133897,0.340428,74.24,1356,406800,"COCOeval bbox, original iscrowd, maxDets=100"
2,lr2e4,2,640,0.145330,0.286614,0.133430,0.065150,0.152704,0.227355,75.88,1349,404700,"COCOeval bbox, original iscrowd, maxDets=100"


,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large,inference_ms_per_image
mean,0.152522,0.302838,0.13556,0.057864,0.142121,0.294192,75.086667
std,0.006473,0.016305,0.00212,0.009876,0.009623,0.059285,0.821300


## 12 — Backup essentials and persistent output
The ZIP includes `best.pt`, metric JSON, evaluation predictions, and training CSVs but **not** redundant `last.pt` / `epoch*.pt` weights. Download the ZIP and save outside the Draft Session. For resumable training, keep each `last.pt` separately (or commit a Kaggle version that persists outputs). Do not run `Save & Run All` just to preserve files.

In [20]:
BACKUP=Path('/kaggle/working/rtdetr_tuning_essentials.zip')
chosen=[]
for p in OUT.rglob('*'):
    if not p.is_file():continue
    if p.name in ('best.pt','completed.json','coco_metrics.json','predictions_coco.json','results.csv','args.yaml','trial_comparison.csv','tuned_3fold_results.csv','tuned_3fold_summary.csv','cv_splits_step6.csv'):
        chosen.append(p)
with zipfile.ZipFile(BACKUP,'w',compression=zipfile.ZIP_STORED) as z:
    for p in chosen:z.write(p,arcname=str(p.relative_to(OUT)))
with zipfile.ZipFile(BACKUP) as z:assert z.testzip() is None
print('Created:',BACKUP,'files:',len(chosen),'size MB:',round(BACKUP.stat().st_size/1e6,1))
print('NOTE: still in temporary /kaggle/working; download or persist via notebook output.')

Created: /kaggle/working/rtdetr_tuning_essentials.zip files: 37 size MB: 898.1
NOTE: still in temporary /kaggle/working; download or persist via notebook output.


In [21]:
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED

RUNS = Path("/kaggle/working/rtdetr_tuning/runs")
BACKUP = Path("/kaggle/working/rtdetr_last_checkpoints.zip")

# Find all last.pt checkpoints
checkpoints = sorted(RUNS.rglob("last.pt"))

assert checkpoints, f"No last.pt checkpoints found in {RUNS}"

print(f"Found {len(checkpoints)} checkpoints:")
for p in checkpoints:
    print(f"  {p.relative_to(RUNS)} ({p.stat().st_size / 1e6:.1f} MB)")

# Create ZIP without recompressing model weights
with ZipFile(BACKUP, "w", compression=ZIP_STORED) as z:
    for p in checkpoints:
        z.write(p, arcname=str(p.relative_to(RUNS)))

# Verify ZIP
with ZipFile(BACKUP) as z:
    assert z.testzip() is None
    archived = set(z.namelist())

assert archived == {str(p.relative_to(RUNS)) for p in checkpoints}

print("\nLAST CHECKPOINTS BACKUP CREATED")
print("Path:", BACKUP)
print("Size:", round(BACKUP.stat().st_size / 1e6, 1), "MB")
print("Files:", len(checkpoints))

Found 4 checkpoints:
  lr2e4_fold_0/weights/last.pt (66.2 MB)
  lr2e4_fold_1/weights/last.pt (66.2 MB)
  lr2e4_fold_2/weights/last.pt (66.2 MB)
  lr5e5_fold_0/weights/last.pt (66.2 MB)

LAST CHECKPOINTS BACKUP CREATED
Path: /kaggle/working/rtdetr_last_checkpoints.zip
Size: 265.0 MB
Files: 4
